# Phase 1: Vietnamese Textbook Extraction on Kaggle

Notebook này chạy Phase 1 theo hướng trong report: dùng DeepSeek-OCR cho OCR/markdown tiếng Việt, dùng MinerU/PDF-Extract-Kit cho layout, ảnh, bảng và metadata. Đầu ra chuẩn hóa về `book.md`, `book.docx`, `rag_chunks.jsonl`, `metadata/book.json`, `metadata/images.json`, `metadata/blocks.jsonl`, và `_previews/images_contact.jpg`.

Khuyến nghị Kaggle: bật GPU `T4 x2`, Internet `On`, rồi chạy lần lượt từ trên xuống. DeepSeek-OCR được chạy data-parallel: chia các trang PDF cho từng GPU để tận dụng cả 2 GPU mà không làm giảm độ chính xác.

In [ ]:
from pathlib import Path
import json, math, os, re, shutil, subprocess, sys, time

# Kaggle th??ng preinstall TensorFlow/pyOpenSSL l?ch version.
# ?p HuggingFace Transformers d?ng PyTorch-only ?? tr?nh l?i OpenSSL ki?u GEN_EMAIL.
os.environ.setdefault('USE_TF', '0')
os.environ.setdefault('TRANSFORMERS_NO_TF', '1')
os.environ.setdefault('USE_FLAX', '0')
os.environ.setdefault('TRANSFORMERS_NO_FLAX', '1')
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '3')

# Thay đường dẫn input của file PDF vào đây
INPUT_PDF = Path('/kaggle/input/vietnam-schoolbooks/SGK Lịch sử và địa lí 6 CD.pdf')

# Fallback khi chạy local trong repo.
if not INPUT_PDF.exists():
    INPUT_PDF = Path('books/SGK Lịch sử và địa lí 6 CD.pdf')

OUTPUT_DIR = Path('/kaggle/working/class_6_phase1') if Path('/kaggle/working').exists() else Path('extracted/class_6_kaggle_phase1')
START_PAGE = 1       # 1-based. Đổi thành 6 để test nhanh Bài 1.
END_PAGE = None      # None = chạy hết sách. Đổi thành 10 để test nhanh.
RENDER_SCALE = 2.4

RUN_MINERU_LAYOUT = True
RUN_DEEPSEEK_OCR = True
RUN_LLM_SPELLCHECK = True
INSTALL_MINERU = True
INSTALL_DEEPSEEK_DEPS = True
ALLOW_LOCAL_FALLBACK = True

# DeepSeek-OCR settings. T4 không hỗ trợ bf16 tốt, notebook sẽ tự chọn fp16 khi cần.
DEEPSEEK_MODEL = 'deepseek-ai/DeepSeek-OCR'
DEEPSEEK_PROMPT = '<image>\n<|grounding|>Convert the document to markdown.'
DEEPSEEK_BASE_SIZE = 1024
DEEPSEEK_IMAGE_SIZE = 640
DEEPSEEK_ATTN_IMPL = 'sdpa'  # ổn định hơn flash_attention_2 trên Kaggle/T4.

# LLM hậu xử lý chính tả/OCR. Có thể đổi sang model khác nếu Kaggle thiếu VRAM.
SPELLCHECK_MODEL = 'Qwen/Qwen3-4B-Instruct-2507'
SPELLCHECK_MAX_SECTION_CHARS = 4200
SPELLCHECK_MAX_NEW_TOKENS = 3072
SPELLCHECK_OVERWRITE_BOOK_MD = True

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('INPUT_PDF =', INPUT_PDF)
print('OUTPUT_DIR =', OUTPUT_DIR)

In [ ]:
def run_cmd(cmd, *, env=None, check=True):
    print('+', ' '.join(map(str, cmd)))
    return subprocess.run(list(map(str, cmd)), env=env, check=check)

run_cmd([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'pip', 'uv'])
run_cmd([sys.executable, '-m', 'pip', 'install', '-q', 'pymupdf', 'pillow', 'opencv-python-headless', 'python-docx', 'tqdm', 'numpy'])

if INSTALL_MINERU:
    # MinerU là wrapper thực dụng cho PDF -> Markdown/JSON và dùng các model từ PDF-Extract-Kit.
    # Nếu cell này lỗi vì CUDA/vLLM dependency, đổi INSTALL_MINERU=False và dùng fallback local ở các cell sau.
    run_cmd(['uv', 'pip', 'install', '--system', '-q', '-U', 'mineru[all]'], check=False)

if INSTALL_DEEPSEEK_DEPS:
    # Không ép flash-attn để tránh compile lâu trên Kaggle T4. Accuracy không phụ thuộc flash-attn.
    # Pin transformers <5 vì MinerU/PDF-Extract-Kit hiện lỗi với transformers 5.x ở PPDocLayoutV2.
    run_cmd([
        sys.executable, '-m', 'pip', 'install', '-q',
        '--force-reinstall', '--no-cache-dir',
        'transformers==4.57.1',
        'huggingface-hub>=0.34.0,<1.0',
        'tokenizers>=0.22,<0.23',
        'accelerate', 'safetensors', 'einops', 'timm', 'sentencepiece'
    ], check=False)

try:
    import torch
    GPU_COUNT = torch.cuda.device_count()
    GPU_IDS = list(range(GPU_COUNT))
except Exception:
    GPU_COUNT = 0
    GPU_IDS = []

CPU_COUNT = os.cpu_count() or 2
os.environ['OMP_NUM_THREADS'] = str(CPU_COUNT)
os.environ['MKL_NUM_THREADS'] = str(CPU_COUNT)
print('CPU_COUNT =', CPU_COUNT)
print('GPU_IDS =', GPU_IDS)

In [ ]:
import fitz
from concurrent.futures import ProcessPoolExecutor, as_completed
from PIL import Image, ImageDraw
from tqdm.auto import tqdm

PAGES_DIR = OUTPUT_DIR / 'pages'
IMAGES_DIR = OUTPUT_DIR / 'images'
METADATA_DIR = OUTPUT_DIR / 'metadata'
PREVIEW_DIR = OUTPUT_DIR / '_previews'
for directory in [PAGES_DIR, IMAGES_DIR, METADATA_DIR, PREVIEW_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

def render_one_page(args):
    pdf_path, page_index, scale, out_dir = args
    doc = fitz.open(str(pdf_path))
    page_no = page_index + 1
    out_path = Path(out_dir) / f'page_{page_no:03d}.jpg'
    if not out_path.exists():
        pix = doc[page_index].get_pixmap(matrix=fitz.Matrix(scale, scale), alpha=False)
        pix.save(str(out_path))
    return {'page_number': page_no, 'path': str(out_path)}

doc = fitz.open(str(INPUT_PDF))
total_pages = len(doc)
start_idx = max(0, START_PAGE - 1)
end_idx = total_pages if END_PAGE is None else min(total_pages, END_PAGE)
page_indexes = list(range(start_idx, end_idx))
print('total_pages =', total_pages, 'selected =', len(page_indexes))

render_jobs = [(INPUT_PDF, i, RENDER_SCALE, PAGES_DIR) for i in page_indexes]
page_images = []
workers = min(CPU_COUNT, max(1, len(render_jobs)))
with ProcessPoolExecutor(max_workers=workers) as executor:
    futures = [executor.submit(render_one_page, job) for job in render_jobs]
    for future in tqdm(as_completed(futures), total=len(futures), desc='Render PDF pages'):
        page_images.append(future.result())
page_images = sorted(page_images, key=lambda item: item['page_number'])
print('rendered pages:', len(page_images))

In [ ]:
MINERU_RAW_DIR = OUTPUT_DIR / '_mineru_raw'
MINERU_RAW_DIR.mkdir(parents=True, exist_ok=True)

def run_mineru():
    if not shutil.which('mineru'):
        print('MinerU CLI not found. Skip MinerU layout stage.')
        return False

    # Kaggle T4 hay lỗi vLLM với backend mặc định `hybrid-engine`.
    # Dùng `pipeline` ổn định hơn; DeepSeek-OCR vẫn đảm nhiệm text chính.
    cmd = [
        'mineru',
        '-p', str(INPUT_PDF),
        '-o', str(MINERU_RAW_DIR),
        '-b', 'pipeline',
        '-m', 'ocr',
        '-f', 'false',
        '-t', 'true',
    ]
    if START_PAGE is not None:
        cmd += ['-s', str(max(0, START_PAGE - 1))]
    if END_PAGE is not None:
        cmd += ['-e', str(max(0, END_PAGE - 1))]
    else:
        print('Warning: END_PAGE=None, MinerU will parse the full PDF.')

    env = os.environ.copy()
    env['USE_TF'] = '0'
    env['TRANSFORMERS_NO_TF'] = '1'
    env['USE_FLAX'] = '0'
    env['TRANSFORMERS_NO_FLAX'] = '1'
    env['TF_CPP_MIN_LOG_LEVEL'] = '3'
    env['MINERU_PROCESSING_WINDOW_SIZE'] = '16'
    env['MINERU_API_MAX_CONCURRENT_REQUESTS'] = '1'
    env['MINERU_PDF_RENDER_THREADS'] = str(min(4, CPU_COUNT))
    if GPU_IDS:
        env['CUDA_VISIBLE_DEVICES'] = ','.join(map(str, GPU_IDS))

    result = run_cmd(cmd, env=env, check=False)
    if result.returncode == 0:
        return True

    print('MinerU pipeline backend failed. Continue without MinerU visuals; DeepSeek OCR can still run.')
    return False

mineru_ok = run_mineru() if RUN_MINERU_LAYOUT else False
print('mineru_ok =', mineru_ok)

def find_mineru_content_list(raw_dir):
    files = sorted(Path(raw_dir).rglob('*_content_list.json'))
    if not files:
        files = sorted(Path(raw_dir).rglob('content_list.json'))
    return files[0] if files else None

def load_mineru_visuals(raw_dir, images_dir):
    content_path = find_mineru_content_list(raw_dir)
    if content_path is None:
        return []
    data = json.loads(content_path.read_text(encoding='utf-8'))
    visuals = []
    for idx, item in enumerate(data):
        item_type = item.get('type')
        if item_type not in {'image', 'table', 'chart'}:
            continue
        page_no = int(item.get('page_idx', 0)) + 1
        src_rel = item.get('img_path') or item.get('image_path')
        rel_out = ''
        if src_rel:
            src = content_path.parent / src_rel
            if not src.exists():
                matches = list(Path(raw_dir).rglob(Path(src_rel).name))
                src = matches[0] if matches else src
            if src.exists():
                suffix = src.suffix or '.jpg'
                dst = images_dir / f'mineru_p{page_no:03d}_{idx:04d}{suffix}'
                shutil.copy2(src, dst)
                rel_out = dst.relative_to(OUTPUT_DIR).as_posix()
        caption = ' '.join(item.get('image_caption') or item.get('table_caption') or item.get('chart_caption') or [])
        visuals.append({
            'type': 'table' if item_type == 'table' else 'image',
            'id': f'{item_type}_{idx:04d}',
            'label': caption[:80] if caption else f'{item_type} {idx}',
            'page': page_no,
            'path': rel_out,
            'caption': caption,
            'bbox': item.get('bbox'),
            'source': 'mineru_pdf_extract_kit',
            'raw_type': item_type,
        })
    return visuals

mineru_visuals = load_mineru_visuals(MINERU_RAW_DIR, IMAGES_DIR) if mineru_ok else []
print('MinerU visual blocks:', len(mineru_visuals))

In [ ]:
DEEPSEEK_OUT_DIR = OUTPUT_DIR / '_deepseek_pages'
DEEPSEEK_OUT_DIR.mkdir(parents=True, exist_ok=True)
WORKER_PATH = OUTPUT_DIR / 'deepseek_ocr_worker.py'

worker_py = r'''
import json, os, sys, traceback
from pathlib import Path

os.environ.setdefault('USE_TF', '0')
os.environ.setdefault('TRANSFORMERS_NO_TF', '1')
os.environ.setdefault('USE_FLAX', '0')
os.environ.setdefault('TRANSFORMERS_NO_FLAX', '1')
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '3')

import torch
from transformers import AutoModel, AutoTokenizer

manifest_path = Path(sys.argv[1])
out_dir = Path(sys.argv[2])
out_dir.mkdir(parents=True, exist_ok=True)
model_name = os.environ.get('DEEPSEEK_MODEL', 'deepseek-ai/DeepSeek-OCR')
prompt = os.environ.get('DEEPSEEK_PROMPT', '<image>\n<|grounding|>Convert the document to markdown.')
attn_impl = os.environ.get('DEEPSEEK_ATTN_IMPL', 'sdpa')
base_size = int(os.environ.get('DEEPSEEK_BASE_SIZE', '1024'))
image_size = int(os.environ.get('DEEPSEEK_IMAGE_SIZE', '640'))

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
try:
    model = AutoModel.from_pretrained(model_name, trust_remote_code=True, use_safetensors=True, _attn_implementation=attn_impl)
except TypeError:
    model = AutoModel.from_pretrained(model_name, trust_remote_code=True, use_safetensors=True)

dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
if torch.cuda.is_available():
    model = model.eval().cuda().to(dtype)
else:
    model = model.eval()

records = [json.loads(line) for line in manifest_path.read_text(encoding='utf-8').splitlines() if line.strip()]
for record in records:
    page_no = int(record['page_number'])
    image_file = record['path']
    result_path = out_dir / f'page_{page_no:03d}.json'
    if result_path.exists():
        continue
    try:
        try:
            res = model.infer(tokenizer, prompt=prompt, image_file=image_file, output_path=str(out_dir), base_size=base_size, image_size=image_size, crop_mode=True, save_results=False, test_compress=True)
        except TypeError:
            res = model.infer(tokenizer, prompt=prompt, image_file=image_file, output_path=str(out_dir), base_size=base_size, image_size=image_size, crop_mode=True, save_results=True, test_compress=True)
        if isinstance(res, str):
            markdown = res
        elif isinstance(res, dict):
            markdown = res.get('text') or res.get('markdown') or json.dumps(res, ensure_ascii=False)
        else:
            markdown = str(res)
        payload = {'page_number': page_no, 'page_image_path': image_file, 'markdown': markdown, 'engine': 'deepseek-ocr'}
    except Exception as exc:
        payload = {'page_number': page_no, 'page_image_path': image_file, 'markdown': '', 'engine': 'deepseek-ocr', 'error': str(exc), 'traceback': traceback.format_exc()}
    result_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')
'''

WORKER_PATH.write_text(worker_py, encoding='utf-8')

def run_deepseek_data_parallel():
    if not RUN_DEEPSEEK_OCR:
        return False
    if not GPU_IDS:
        print('No GPU found. Skip DeepSeek-OCR.')
        return False
    shards = [[] for _ in GPU_IDS]
    for idx, rec in enumerate(page_images):
        shards[idx % len(GPU_IDS)].append(rec)
    processes = []
    for gpu_id, shard in zip(GPU_IDS, shards):
        manifest = OUTPUT_DIR / f'deepseek_manifest_gpu{gpu_id}.jsonl'
        manifest.write_text('\n'.join(json.dumps(item, ensure_ascii=False) for item in shard), encoding='utf-8')
        env = os.environ.copy()
        env['USE_TF'] = '0'
        env['TRANSFORMERS_NO_TF'] = '1'
        env['USE_FLAX'] = '0'
        env['TRANSFORMERS_NO_FLAX'] = '1'
        env['TF_CPP_MIN_LOG_LEVEL'] = '3'
        env['CUDA_VISIBLE_DEVICES'] = str(gpu_id)
        env['DEEPSEEK_MODEL'] = DEEPSEEK_MODEL
        env['DEEPSEEK_PROMPT'] = DEEPSEEK_PROMPT
        env['DEEPSEEK_ATTN_IMPL'] = DEEPSEEK_ATTN_IMPL
        env['DEEPSEEK_BASE_SIZE'] = str(DEEPSEEK_BASE_SIZE)
        env['DEEPSEEK_IMAGE_SIZE'] = str(DEEPSEEK_IMAGE_SIZE)
        cmd = [sys.executable, str(WORKER_PATH), str(manifest), str(DEEPSEEK_OUT_DIR)]
        print('+ GPU', gpu_id, 'pages', len(shard))
        processes.append(subprocess.Popen(cmd, env=env))
    ok = True
    for process in processes:
        ok = (process.wait() == 0) and ok
    return ok

deepseek_ok = run_deepseek_data_parallel()
print('deepseek_ok =', deepseek_ok)

In [ ]:
def load_deepseek_pages():
    pages = {}
    for path in sorted(DEEPSEEK_OUT_DIR.glob('page_*.json')):
        data = json.loads(path.read_text(encoding='utf-8'))
        pages[int(data['page_number'])] = data
    return pages

def normalize_text(value):
    import unicodedata
    value = unicodedata.normalize('NFD', value.lower())
    value = ''.join(ch for ch in value if unicodedata.category(ch) != 'Mn')
    value = value.replace('đ', 'd')
    return re.sub(r'\s+', ' ', value).strip()

def figure_key(text):
    match = re.search(r'h(?:ì|i)nh\s+(\d{1,2})\s*[\.,]\s*(\d{1,2})', text, flags=re.IGNORECASE)
    if not match:
        match = re.search(r'hinh\s+(\d{1,2})\s*[\.,]\s*(\d{1,2})', normalize_text(text))
    return f'{int(match.group(1))}_{int(match.group(2))}' if match else None

def image_markdown(block):
    alt = block.get('caption') or block.get('label') or block.get('id')
    return f'![{alt}]({block.get("path", "")})'

def insert_images_into_lines(page_no, markdown, visuals):
    lines = [line.rstrip() for line in (markdown or '').splitlines()]
    page_visuals = [dict(v) for v in visuals if int(v.get('page', -1)) == int(page_no) and v.get('path')]
    for visual in page_visuals:
        visual['figure_key'] = figure_key(visual.get('caption', '') or visual.get('label', ''))
        visual['used'] = False
    output = []
    blocks = []
    order = 0
    for line in lines:
        norm_line = normalize_text(line)
        line_key = figure_key(line)
        for visual in page_visuals:
            if visual['used']:
                continue
            caption_norm = normalize_text(visual.get('caption', ''))
            hit = visual.get('figure_key') and visual.get('figure_key') == line_key
            hit = hit or (caption_norm and len(caption_norm) > 8 and caption_norm[:40] in norm_line)
            if hit:
                output.append(image_markdown(visual))
                blocks.append({**visual, 'order': order})
                order += 1
                visual['used'] = True
        if line.strip():
            output.append(line)
            blocks.append({'type': 'text', 'order': order, 'text': line, 'page': page_no, 'source': 'deepseek-ocr'})
            order += 1
    for visual in page_visuals:
        if not visual['used']:
            output.append(image_markdown(visual))
            blocks.append({**visual, 'order': order})
            order += 1
    return output, blocks

def chunk_text(text, chunk_size=1800, overlap=200):
    text = re.sub(r'\s+', ' ', text).strip()
    if not text:
        return []
    chunks, start = [], 0
    while start < len(text):
        end = min(len(text), start + chunk_size)
        chunks.append(text[start:end].strip())
        if end >= len(text):
            break
        start = max(0, end - overlap)
    return chunks

deepseek_pages = load_deepseek_pages()
if not deepseek_pages and ALLOW_LOCAL_FALLBACK and Path('scripts/extract_schoolbook.py').exists():
    print('DeepSeek output not found. Running local fallback extractor to keep pipeline usable.')
    fallback_out = OUTPUT_DIR / '_fallback_local'
    cmd = [sys.executable, 'scripts/extract_schoolbook.py', '--pdf', str(INPUT_PDF), '--out', str(fallback_out), '--start-page', str(START_PAGE), '--scale', str(RENDER_SCALE)]
    if END_PAGE is not None:
        cmd += ['--end-page', str(END_PAGE)]
    run_cmd(cmd, check=True)
    for name in ['book.md', 'book.docx', 'rag_chunks.jsonl']:
        src = fallback_out / name
        if src.exists():
            shutil.copy2(src, OUTPUT_DIR / name)
    for name in ['images', 'metadata', 'pages', '_previews']:
        src = fallback_out / name
        if src.exists():
            shutil.copytree(src, OUTPUT_DIR / name, dirs_exist_ok=True)
else:
    book_lines = [f'# {INPUT_PDF.stem}', '', f'> Source PDF: `{INPUT_PDF}`', f'> Generated at: `{time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())}`', '']
    page_records = []
    block_records = []
    rag_records = []
    for page in page_images:
        page_no = int(page['page_number'])
        page_md = deepseek_pages.get(page_no, {}).get('markdown', '')
        if not page_md:
            page_md = f'[OCR missing for page {page_no}]'
        page_lines, blocks = insert_images_into_lines(page_no, page_md, mineru_visuals)
        book_lines += [f'## PDF Page {page_no}', '', *page_lines, '', '</break>', '']
        page_text = '\n'.join(line for line in page_lines if not line.startswith('!['))
        page_record = {'source_pdf': str(INPUT_PDF), 'page_number': page_no, 'page_image_path': page['path'], 'text': page_text, 'text_blocks': blocks}
        page_records.append(page_record)
        (METADATA_DIR / 'pages').mkdir(parents=True, exist_ok=True)
        (METADATA_DIR / 'pages' / f'page_{page_no:03d}.json').write_text(json.dumps(page_record, ensure_ascii=False, indent=2), encoding='utf-8')
        for block in blocks:
            block_records.append({**block, 'page_number': page_no})
        images_for_page = [{k: v for k, v in block.items() if k in {'id', 'path', 'caption', 'label', 'type'}} for block in blocks if block.get('type') in {'image', 'table'}]
        for idx, chunk in enumerate(chunk_text(page_text), start=1):
            rag_records.append({'chunk_id': f'page_{page_no:03d}_{idx:02d}', 'source_pdf': str(INPUT_PDF), 'page_number': page_no, 'text': chunk, 'images': images_for_page})

    (OUTPUT_DIR / 'book.md').write_text('\n'.join(book_lines).strip() + '\n', encoding='utf-8')
    image_records = [block for block in block_records if block.get('type') in {'image', 'table'}]
    (METADATA_DIR / 'images.json').write_text(json.dumps(image_records, ensure_ascii=False, indent=2), encoding='utf-8')
    (METADATA_DIR / 'blocks.jsonl').write_text('\n'.join(json.dumps(x, ensure_ascii=False) for x in block_records) + '\n', encoding='utf-8')
    (OUTPUT_DIR / 'rag_chunks.jsonl').write_text('\n'.join(json.dumps(x, ensure_ascii=False) for x in rag_records) + '\n', encoding='utf-8')
    summary = {'source_pdf': str(INPUT_PDF), 'pages_processed': len(page_records), 'ocr_engine': 'deepseek-ocr', 'layout_engine': 'mineru/pdf-extract-kit' if mineru_visuals else 'none', 'stats': {'blocks': len(block_records), 'images': len(image_records), 'rag_chunks': len(rag_records)}, 'outputs': {'markdown': 'book.md', 'docx': 'book.docx', 'rag_chunks': 'rag_chunks.jsonl', 'page_metadata_dir': 'metadata/pages', 'image_metadata': 'metadata/images.json'}}
    (METADATA_DIR / 'book.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')

print('Merge done:', OUTPUT_DIR)

In [ ]:
# Optional LLM spell-check stage theo report: Qwen local, không gọi API.
# Stage này giữ lại book_raw_ocr.md để audit, rồi ghi bản đã sửa vào book.md/book_corrected.md.
SPELLCHECK_SCRIPT = Path('scripts/spellcheck_markdown_qwen.py')
if not SPELLCHECK_SCRIPT.exists():
    SPELLCHECK_SCRIPT.parent.mkdir(parents=True, exist_ok=True)
    EMBEDDED_SPELLCHECK_SCRIPT = r'''from __future__ import annotations

import argparse
import json
import os
import re
import subprocess
import sys

os.environ.setdefault("USE_TF", "0")
os.environ.setdefault("TRANSFORMERS_NO_TF", "1")
os.environ.setdefault("USE_FLAX", "0")
os.environ.setdefault("TRANSFORMERS_NO_FLAX", "1")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable


IMAGE_TOKEN_RE = re.compile(r"^!\[.*?\]\(.*?\)\s*$")
PAGE_HEADING_RE = re.compile(r"(?m)^## PDF Page\s+(\d+)\s*$")


@dataclass
class PageSection:
    page_number: int
    text: str


def split_page_sections(markdown: str) -> tuple[str, list[PageSection]]:
    matches = list(PAGE_HEADING_RE.finditer(markdown))
    if not matches:
        return "", [PageSection(page_number=1, text=markdown)]

    preamble = markdown[: matches[0].start()].rstrip() + "\n\n"
    sections: list[PageSection] = []
    for index, match in enumerate(matches):
        end = matches[index + 1].start() if index + 1 < len(matches) else len(markdown)
        sections.append(PageSection(page_number=int(match.group(1)), text=markdown[match.start() : end].strip()))
    return preamble, sections


def protect_markdown_lines(text: str) -> tuple[str, dict[str, str]]:
    placeholders: dict[str, str] = {}
    protected_lines: list[str] = []
    for line in text.splitlines():
        if IMAGE_TOKEN_RE.match(line.strip()):
            token = f"@@IMAGE_LINK_{len(placeholders):04d}@@"
            placeholders[token] = line
            protected_lines.append(token)
        else:
            protected_lines.append(line)
    return "\n".join(protected_lines), placeholders


def restore_markdown_lines(text: str, placeholders: dict[str, str]) -> str:
    for token, original in placeholders.items():
        text = text.replace(token, original)
    return text


def strip_code_fence(text: str) -> str:
    text = text.strip()
    if text.startswith("```"):
        text = re.sub(r"^```(?:markdown|md|text)?\s*", "", text, flags=re.IGNORECASE)
        text = re.sub(r"\s*```$", "", text)
    return text.strip()


def split_long_text(text: str, max_chars: int) -> list[str]:
    if len(text) <= max_chars:
        return [text]

    parts: list[str] = []
    current: list[str] = []
    current_len = 0
    for paragraph in re.split(r"(\n\s*\n)", text):
        if current_len + len(paragraph) > max_chars and current:
            parts.append("".join(current).strip())
            current = []
            current_len = 0
        current.append(paragraph)
        current_len += len(paragraph)
    if current:
        parts.append("".join(current).strip())
    return [part for part in parts if part]


def load_qwen(model_name: str):
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer

    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
    kwargs = {
        "trust_remote_code": True,
        "torch_dtype": dtype,
        "device_map": "auto",
    }
    try:
        model = AutoModelForCausalLM.from_pretrained(model_name, attn_implementation="sdpa", **kwargs)
    except TypeError:
        model = AutoModelForCausalLM.from_pretrained(model_name, **kwargs)
    model.eval()
    return tokenizer, model


def generate_text(tokenizer, model, protected_markdown: str, max_new_tokens: int) -> str:
    import torch

    messages = [
        {
            "role": "system",
            "content": (
                "Bạn là bộ hậu xử lý OCR tiếng Việt cho sách giáo khoa. "
                "Chỉ sửa lỗi OCR, chính tả, dấu tiếng Việt, ký tự rác và lỗi tách từ. "
                "Không thêm kiến thức mới, không diễn giải, không tóm tắt, không đổi số liệu hoặc tên riêng nếu không chắc."
            ),
        },
        {
            "role": "user",
            "content": (
                "Sửa đoạn Markdown OCR dưới đây.\n"
                "Yêu cầu bắt buộc:\n"
                "- Giữ nguyên heading Markdown, số thứ tự câu hỏi, `</break>` và các placeholder dạng @@IMAGE_LINK_0000@@.\n"
                "- Không xóa hoặc thêm ảnh, không đổi đường dẫn ảnh.\n"
                "- Output duy nhất là Markdown đã sửa, không giải thích.\n\n"
                f"```markdown\n{protected_markdown}\n```"
            ),
        },
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            do_sample=False,
            max_new_tokens=max_new_tokens,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated = outputs[0][inputs["input_ids"].shape[-1] :]
    return strip_code_fence(tokenizer.decode(generated, skip_special_tokens=True))


def correct_section(tokenizer, model, text: str, max_section_chars: int, max_new_tokens: int) -> str:
    corrected_parts: list[str] = []
    for part in split_long_text(text, max_section_chars):
        corrected_parts.append(generate_text(tokenizer, model, part, max_new_tokens))
    return "\n\n".join(corrected_parts).strip()


def worker_main(args: argparse.Namespace) -> None:
    tokenizer, model = load_qwen(args.model)
    out_dir = Path(args.worker_output)
    out_dir.mkdir(parents=True, exist_ok=True)

    manifest = Path(args.worker_manifest)
    records = [json.loads(line) for line in manifest.read_text(encoding="utf-8").splitlines() if line.strip()]
    for record in records:
        page_number = int(record["page_number"])
        out_path = out_dir / f"page_{page_number:03d}.json"
        if out_path.exists() and not args.force:
            continue
        try:
            corrected = correct_section(
                tokenizer,
                model,
                str(record["protected_text"]),
                args.max_section_chars,
                args.max_new_tokens,
            )
            payload = {"page_number": page_number, "corrected_protected_text": corrected}
        except Exception as exc:
            payload = {"page_number": page_number, "corrected_protected_text": record["protected_text"], "error": str(exc)}
        out_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")


def write_jsonl(path: Path, records: Iterable[dict[str, object]]) -> None:
    path.write_text("\n".join(json.dumps(record, ensure_ascii=False) for record in records) + "\n", encoding="utf-8")


def markdown_text_only(section: str) -> str:
    lines: list[str] = []
    for line in section.splitlines():
        stripped = line.strip()
        if not stripped or stripped.startswith("!["):
            continue
        if stripped == "</break>":
            continue
        stripped = re.sub(r"^#{1,6}\s*", "", stripped)
        lines.append(stripped)
    return re.sub(r"\s+", " ", " ".join(lines)).strip()


def chunk_text(text: str, chunk_size: int, overlap: int) -> list[str]:
    text = re.sub(r"\s+", " ", text).strip()
    if not text:
        return []
    chunks: list[str] = []
    start = 0
    while start < len(text):
        end = min(len(text), start + chunk_size)
        chunks.append(text[start:end].strip())
        if end >= len(text):
            break
        start = max(0, end - overlap)
    return chunks


def spawn_gpu_workers(
    sections: list[dict[str, object]],
    args: argparse.Namespace,
    work_dir: Path,
    gpu_ids: list[str],
) -> None:
    shard_count = max(1, len(gpu_ids))
    shards: list[list[dict[str, object]]] = [[] for _ in range(shard_count)]
    for index, section in enumerate(sections):
        shards[index % shard_count].append(section)

    processes: list[subprocess.Popen[bytes]] = []
    for shard_index, shard in enumerate(shards):
        manifest_path = work_dir / f"spellcheck_manifest_{shard_index}.jsonl"
        write_jsonl(manifest_path, shard)
        env = os.environ.copy()
        env["USE_TF"] = "0"
        env["TRANSFORMERS_NO_TF"] = "1"
        env["USE_FLAX"] = "0"
        env["TRANSFORMERS_NO_FLAX"] = "1"
        env["TF_CPP_MIN_LOG_LEVEL"] = "3"
        if gpu_ids:
            env["CUDA_VISIBLE_DEVICES"] = gpu_ids[shard_index]
        command = [
            sys.executable,
            str(Path(__file__).resolve()),
            "--worker-manifest",
            str(manifest_path),
            "--worker-output",
            str(args.worker_output),
            "--model",
            args.model,
            "--max-section-chars",
            str(args.max_section_chars),
            "--max-new-tokens",
            str(args.max_new_tokens),
        ]
        if args.force:
            command.append("--force")
        processes.append(subprocess.Popen(command, env=env))

    failed: list[int] = []
    for process in processes:
        exit_code = process.wait()
        if exit_code != 0:
            failed.append(exit_code)
    if failed:
        raise RuntimeError(f"Spellcheck worker failed: {failed}")


def update_metadata(
    metadata_dir: Path,
    corrected_sections: list[PageSection],
    model_name: str,
    chunk_size: int,
    overlap: int,
    rag_output: Path,
) -> None:
    pages_dir = metadata_dir / "pages"
    corrected_by_page = {section.page_number: section.text for section in corrected_sections}
    for page_number, section in corrected_by_page.items():
        page_path = pages_dir / f"page_{page_number:03d}.json"
        if not page_path.exists():
            continue
        payload = json.loads(page_path.read_text(encoding="utf-8"))
        payload["text_corrected"] = markdown_text_only(section)
        payload["spellcheck_engine"] = model_name
        page_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")

    rag_records: list[dict[str, object]] = []
    image_records_path = metadata_dir / "images.json"
    image_records = json.loads(image_records_path.read_text(encoding="utf-8")) if image_records_path.exists() else []
    images_by_page: dict[int, list[dict[str, object]]] = {}
    for record in image_records:
        page_number = int(record.get("page_number") or record.get("page") or 0)
        images_by_page.setdefault(page_number, []).append(
            {
                "id": record.get("id"),
                "path": record.get("path"),
                "label": record.get("label"),
                "caption": record.get("caption"),
                "type": record.get("type"),
            }
        )

    for section in corrected_sections:
        text = markdown_text_only(section.text)
        for index, chunk in enumerate(chunk_text(text, chunk_size, overlap), start=1):
            rag_records.append(
                {
                    "chunk_id": f"page_{section.page_number:03d}_{index:02d}",
                    "page_number": section.page_number,
                    "text": chunk,
                    "images": images_by_page.get(section.page_number, []),
                    "spellcheck_engine": model_name,
                }
            )
    write_jsonl(rag_output, rag_records)

    book_json_path = metadata_dir / "book.json"
    if book_json_path.exists():
        summary = json.loads(book_json_path.read_text(encoding="utf-8"))
        summary["spellcheck"] = {
            "enabled": True,
            "model": model_name,
            "pages_corrected": len(corrected_sections),
            "rag_chunks_after_spellcheck": len(rag_records),
        }
        book_json_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")


def main() -> None:
    parser = argparse.ArgumentParser(description="Post-process OCR Markdown with a local Qwen LLM.")
    parser.add_argument("--input", type=Path, help="Input Markdown file.")
    parser.add_argument("--output", type=Path, help="Corrected Markdown output.")
    parser.add_argument("--raw-output", type=Path, default=None, help="Where to keep the raw OCR Markdown.")
    parser.add_argument("--metadata-dir", type=Path, default=None, help="Metadata directory to update.")
    parser.add_argument("--rag-output", type=Path, default=None, help="RAG chunks JSONL output.")
    parser.add_argument("--model", default="Qwen/Qwen3-4B-Instruct-2507")
    parser.add_argument("--gpus", default="auto", help="Comma-separated GPU IDs, auto, or none.")
    parser.add_argument("--max-section-chars", type=int, default=4200)
    parser.add_argument("--max-new-tokens", type=int, default=3072)
    parser.add_argument("--chunk-size", type=int, default=1800)
    parser.add_argument("--chunk-overlap", type=int, default=200)
    parser.add_argument("--force", action="store_true")
    parser.add_argument("--worker-manifest", type=Path, default=None)
    parser.add_argument("--worker-output", type=Path, default=None)
    args = parser.parse_args()

    if args.worker_manifest:
        worker_main(args)
        return

    if not args.input or not args.output:
        raise ValueError("--input and --output are required outside worker mode.")

    raw_markdown = args.input.read_text(encoding="utf-8")
    if args.raw_output and not args.raw_output.exists():
        args.raw_output.write_text(raw_markdown, encoding="utf-8")

    preamble, page_sections = split_page_sections(raw_markdown)
    protected_records: list[dict[str, object]] = []
    placeholder_map: dict[int, dict[str, str]] = {}
    for section in page_sections:
        protected_text, placeholders = protect_markdown_lines(section.text)
        placeholder_map[section.page_number] = placeholders
        protected_records.append({"page_number": section.page_number, "protected_text": protected_text})

    work_dir = args.output.parent / "_spellcheck_qwen"
    worker_output = work_dir / "pages"
    worker_output.mkdir(parents=True, exist_ok=True)
    args.worker_output = worker_output

    if args.gpus == "none":
        gpu_ids: list[str] = []
    elif args.gpus == "auto":
        try:
            import torch

            gpu_ids = [str(index) for index in range(torch.cuda.device_count())]
        except Exception:
            gpu_ids = []
    else:
        gpu_ids = [value.strip() for value in args.gpus.split(",") if value.strip()]

    if gpu_ids:
        spawn_gpu_workers(protected_records, args, work_dir, gpu_ids)
    else:
        manifest_path = work_dir / "spellcheck_manifest_cpu.jsonl"
        write_jsonl(manifest_path, protected_records)
        args.worker_manifest = manifest_path
        args.worker_output = worker_output
        worker_main(args)

    corrected_sections: list[PageSection] = []
    for section in page_sections:
        corrected_path = worker_output / f"page_{section.page_number:03d}.json"
        if corrected_path.exists():
            payload = json.loads(corrected_path.read_text(encoding="utf-8"))
            corrected_protected = str(payload.get("corrected_protected_text") or section.text)
        else:
            corrected_protected = section.text
        corrected = restore_markdown_lines(corrected_protected, placeholder_map[section.page_number])
        corrected_sections.append(PageSection(page_number=section.page_number, text=corrected))

    corrected_markdown = preamble + "\n\n".join(section.text.strip() for section in corrected_sections).strip() + "\n"
    args.output.write_text(corrected_markdown, encoding="utf-8")

    if args.metadata_dir and args.rag_output:
        update_metadata(args.metadata_dir, corrected_sections, args.model, args.chunk_size, args.chunk_overlap, args.rag_output)

    print(f"Spellcheck done: {args.output}")
    print(f"Pages corrected: {len(corrected_sections)}")


if __name__ == "__main__":
    main()
'''
    SPELLCHECK_SCRIPT.write_text(EMBEDDED_SPELLCHECK_SCRIPT, encoding='utf-8')
    print('Created embedded spellcheck script:', SPELLCHECK_SCRIPT)
if RUN_LLM_SPELLCHECK:
    if not SPELLCHECK_SCRIPT.exists():
        print('Spellcheck script not found. Skip LLM correction.')
    else:
        corrected_md = OUTPUT_DIR / 'book_corrected.md'
        cmd = [
            sys.executable,
            str(SPELLCHECK_SCRIPT),
            '--input', str(OUTPUT_DIR / 'book.md'),
            '--output', str(corrected_md),
            '--raw-output', str(OUTPUT_DIR / 'book_raw_ocr.md'),
            '--metadata-dir', str(METADATA_DIR),
            '--rag-output', str(OUTPUT_DIR / 'rag_chunks.jsonl'),
            '--model', SPELLCHECK_MODEL,
            '--gpus', 'auto',
            '--max-section-chars', str(SPELLCHECK_MAX_SECTION_CHARS),
            '--max-new-tokens', str(SPELLCHECK_MAX_NEW_TOKENS),
            '--chunk-size', '1800',
            '--chunk-overlap', '200',
        ]
        env = os.environ.copy()
        env['USE_TF'] = '0'
        env['TRANSFORMERS_NO_TF'] = '1'
        env['USE_FLAX'] = '0'
        env['TRANSFORMERS_NO_FLAX'] = '1'
        env['TF_CPP_MIN_LOG_LEVEL'] = '3'
        result = run_cmd(cmd, env=env, check=False)
        if result.returncode == 0 and corrected_md.exists() and SPELLCHECK_OVERWRITE_BOOK_MD:
            shutil.copy2(corrected_md, OUTPUT_DIR / 'book.md')
            print('Spellcheck applied to book.md')
        elif result.returncode != 0:
            print('Spellcheck failed. Keep raw OCR book.md.')
else:
    print('RUN_LLM_SPELLCHECK=False. Skip LLM correction.')


In [ ]:
from docx import Document
from docx.shared import Inches

def make_docx():
    book_md = OUTPUT_DIR / 'book.md'
    if not book_md.exists():
        return False
    doc = Document()
    doc.add_heading(INPUT_PDF.stem, level=1)
    for line in book_md.read_text(encoding='utf-8').splitlines():
        if line.startswith('# '):
            continue
        if line.startswith('## '):
            doc.add_heading(line[3:].strip(), level=2)
        elif line.startswith('### '):
            doc.add_heading(line[4:].strip(), level=3)
        elif line.startswith('!['):
            match = re.search(r'\]\((.*?)\)', line)
            if match:
                image_path = OUTPUT_DIR / match.group(1)
                if image_path.exists():
                    try:
                        doc.add_picture(str(image_path), width=Inches(5.6))
                    except Exception:
                        doc.add_paragraph(str(image_path))
        elif line.strip():
            doc.add_paragraph(line)
    doc.save(str(OUTPUT_DIR / 'book.docx'))
    return True

def make_contact_sheet():
    images_json = METADATA_DIR / 'images.json'
    if not images_json.exists():
        return None
    records = json.loads(images_json.read_text(encoding='utf-8'))
    records = [r for r in records if r.get('path')]
    if not records:
        return None
    thumb_w, label_h, pad, cols = 260, 42, 12, 3
    rows = math.ceil(len(records) / cols)
    sheet = Image.new('RGB', (cols * thumb_w + (cols + 1) * pad, rows * (thumb_w + label_h) + (rows + 1) * pad), 'white')
    draw = ImageDraw.Draw(sheet)
    for idx, rec in enumerate(records):
        path = OUTPUT_DIR / rec['path']
        if not path.exists():
            continue
        with Image.open(path).convert('RGB') as img:
            img.thumbnail((thumb_w, thumb_w - label_h))
            col, row = idx % cols, idx // cols
            x = pad + col * (thumb_w + pad)
            y = pad + row * (thumb_w + label_h + pad)
            sheet.paste(img, (x + (thumb_w - img.width) // 2, y))
            draw.text((x + 4, y + thumb_w - 12), f'{rec.get("id")} p.{rec.get("page_number") or rec.get("page")}', fill=(20, 20, 20))
    out = PREVIEW_DIR / 'images_contact.jpg'
    sheet.save(out, quality=88)
    return out

print('docx:', make_docx())
contact = make_contact_sheet()
print('contact:', contact)
for path in [OUTPUT_DIR / 'book.md', OUTPUT_DIR / 'book.docx', OUTPUT_DIR / 'rag_chunks.jsonl', METADATA_DIR / 'book.json', METADATA_DIR / 'images.json', contact]:
    if path and Path(path).exists():
        print(path, Path(path).stat().st_size)

In [ ]:
try:
    from IPython.display import display, Image as IPImage
    if contact and Path(contact).exists():
        display(IPImage(filename=str(contact)))
except Exception as exc:
    print(exc)

print('\nFinal output directory:')
print(OUTPUT_DIR)
print('\nFiles:')
for path in sorted(OUTPUT_DIR.rglob('*')):
    if path.is_file() and path.suffix.lower() in {'.md', '.docx', '.json', '.jsonl', '.jpg'}:
        print(path.relative_to(OUTPUT_DIR))